In [1]:
import os
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from sklearn.svm import SVC

for i in range(1, 9):
    folder_path = os.path.join('initial_data', f'function_{i}')
    inputs_path = os.path.join(folder_path, 'initial_inputs.npy')
    outputs_path = os.path.join(folder_path, 'initial_outputs.npy')
    
    if not os.path.exists(inputs_path) or not os.path.exists(outputs_path):
        continue
        
    X_f = np.load(inputs_path)
    y_f = np.load(outputs_path)
    
    dim = X_f.shape[1] if len(X_f.shape) > 1 else 1
    
    if i == 5:
        beta = 0.01
        alpha_val = 1e-4
    elif i == 8:
        beta = 1.5
        alpha_val = 1e-4
    elif i in [1, 2]:
        beta = 4.5
        alpha_val = 1e-3
    else:
        beta = 1.5
        alpha_val = 1e-4
        
    np.random.seed(800 + i)
    X_grid = np.random.uniform(0.0, 1.0, size=(60000, dim))
    
    try:
        opt = 'fmin_l_bfgs_b' if i == 8 else None
        kernel = Matern(length_scale=[0.2] * dim, length_scale_bounds=(1e-2, 1e2), nu=2.5) if i == 8 else Matern(length_scale=[0.2] * dim, nu=2.5)
        gp = GaussianProcessRegressor(kernel=kernel, alpha=alpha_val, normalize_y=True, optimizer=opt)
        gp.fit(X_f, y_f)
        
        if i in [3, 5, 7] and len(y_f) >= 15:
            threshold = np.percentile(y_f, 75)
            y_labels = np.where(y_f >= threshold, 1, 0)
            
            if len(np.unique(y_labels)) > 1:
                svm = SVC(kernel='rbf', C=1.0, gamma='scale')
                svm.fit(X_f, y_labels)
                preds = svm.predict(X_grid)
                X_filtered = X_grid[preds == 1]
                
                if len(X_filtered) > 100:
                    X_grid = X_filtered
        
        mean, sigma = gp.predict(X_grid, return_std=True)
        ucb_values = mean + beta * sigma
        best_idx = np.argmax(ucb_values)
        next_point = X_grid[best_idx]
        
    except Exception:
        from scipy.spatial.distance import cdist
        distances = cdist(X_grid, X_f)
        min_distances = np.min(distances, axis=1)
        best_idx = np.argmax(min_distances)
        next_point = X_grid[best_idx]
    
    portal_string = "-".join([f"{val:.6f}" for val in next_point])
    print(portal_string)


0.002955-0.999529
0.601989-0.467055
0.361819-0.708576-0.534116
0.336502-0.392757-0.338087-0.436308
0.330949-0.881220-0.969812-0.995357
0.638413-0.175165-0.656327-0.795929-0.197718
0.133972-0.382382-0.205471-0.268391-0.253368-0.792971
0.065882-0.123709-0.139637-0.052027-0.925617-0.137964-0.201843-0.365258


C:\Users\Public\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
